# Morfessor model generator

developed by Kow Kuroda

created on 2025/02/15
modified on 2025/02/17

# functions

In [ ]:
def get_files(target_lang, check: bool = True):
    ## select data files
    import glob
    data_files = glob.glob(f"../data/inflected/{target_lang}/*")
    data_files = sorted([ file for file in data_files if ".csv" in file ])
    if check:
        import pprint as pp
        pp.pprint(data_files)
    return data_files

In [ ]:
def process_ske_lines (lines, form_dict_raw, form_counter, uncapitalize: bool = True, l_splitter: str = ",", f_splitter: str = ",", tag_cleaner: str = r'[*"]', word_reg: str = r"\w+", check: bool = False):
    "process Sketch Engine sample data to get token/pos pairs"
    import re
    counter = 0
    for i, line in enumerate (lines):
        if check:
            print(f"line {i:4d}")
        fields = line.split(l_splitter)
        if check:
            print(f"len(fields): {len(fields)}")
        if len (fields) < 6:
            continue
        ## main
        for field in fields:
            blocks = field.split (f_splitter)
            for block in blocks:
                if check:
                    print (f"block: {block}")
                try:
                    tokens = block.split ()
                    for token in tokens:
                        form, tag = token.split("/")
                        form      = form.strip()
                        ## uncapitalize form
                        if uncapitalize:
                            form      = form.strip().lower()
                        ## process POS tag
                        tag = re.sub (tag_cleaner, '', tag.strip())
                        if re.match (word_reg, form):
                            form_counter [form] += 1
                            ##
                            counter += 1
                            print (f"form {counter:04d} <{form}> with tag <{tag}> registered")
                            if tag not in form_dict_raw [form]: 
                                form_dict_raw [form].append (tag)
                except ValueError:
                    pass

In [ ]:
def process_files(data_files, check: bool = True):
    "process words in data"
    import io, re
    import collections
    form_dict_raw = collections.defaultdict(list)
    form_counter  = collections.defaultdict(int)
    ##
    for file in data_files:
        print(f"opening {file}")
        with io.open(file, encoding = 'utf-8_sig') as word:
            lines = word.readlines()
            process_ske_lines (lines, form_dict_raw = form_dict_raw, form_counter = form_counter)
    ##
    if check:
        print(form_dict_raw)
        print(form_counter)
    ##
    return form_dict_raw, form_counter

In [ ]:
def log_transform(x):
    import math
    return int(round(math.log(x + 1, 2)))

In [ ]:
def generate_Morfessor_models(train_data, target_lang, algorithm, threshold):
    "generate Morfessor models with different morph_lengths"
    ## define model
    import morfessor as morf
    model_tokens = morf.BaselineModel(use_skips = True)
    model_tokens.load_data(train_data, count_modifier = log_transform)

    ## loop over morph_sizes
    gen_count = 0
    import numpy as np
    morph_sizes = list(np.arange(2.0, 3.5, 0.5))
    for morph_size in morph_sizes:
        print(f"============================")
        print(f"generating model with morph_size: {morph_size}")
        # set morphlength parameter
        model_tokens.set_corpus_weight_updater(morf.MorphLengthCorpusWeight(morph_size, threshold))
        # define target file
        model_file = f"models/model-{target_lang}-{algorithm}-ml{morph_size}.bin"
        mio = morf.MorfessorIO()
        import os
        if os.path.exists(model_file):
            print(f"target file exists; skipped generation")
        else:
            # train
            print(f"training {model_file}")
            model_tokens.train_batch(algorithm = algorithm)
            # show result
            print(f"segmentations:")
            print(list(model_tokens.get_segmentations()))
            # write out
            mio.write_binary_file(model_file, model_tokens)
            gen_count += 1
    ##
    return gen_count


# main

In [ ]:
target_langs = ['Czech', 'French', 'German', 'Irish']
for target_lang in target_langs:
    ## get files and generate base data structures
    data_files = get_files (target_lang)
    form_dict_raw, form_counter = process_files (data_files)
    
    ## generate models
    freq_form_pairs = [ f"{freq} {form}" for form, freq in
                       sorted (form_counter.items(), key = lambda x: x[-1], reverse = True) ]
    
    ## generate train_data with specific format
    train_data = [ (int(pair[0]), pair[1]) for pair in map(lambda x: x.split(), freq_form_pairs) ]
    
    # parameters
    algorithms = [ 'recursive', 'viterbi' ]
    algorithm  = algorithms[0]
    threshold  = 0.01
    gen_count = generate_Morfessor_models(train_data, target_lang, algorithm, threshold)
##
print(f"generation of {gen_count} done")